# 03_embed_texts.ipynb

Embeds every turn of every dialogue. `*_embed.json` holds the victim embeddings (e5-base-v2) that the attack notebooks treat as leaked; `*_localembed.json` holds all-mpnet-base-v2 embeddings used by the TCFN notebooks.

Paths are relative: `data/` holds dialogues and embeddings (not included). See the README.


In [ ]:
import json
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "data"
DATASETS = ["bank", "ill", "new_user_profile"]  # financial, medical, user profile

# victim embedder: produces the "leaked" embeddings the attacker sees
victim_model = SentenceTransformer("intfloat/e5-base-v2", device=DEVICE)
# local embedder used by the TCFN notebooks (the sliding-window notebooks embed candidates on the fly)
local_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)


In [ ]:
def convert(model, path_in, path_out, batch_size=32):
    all_dialogue_embeddings = []
    with open(path_in, "r") as f:
        dataset = json.load(f)
    for data_group in tqdm(dataset, desc="Processing Groups"):
        texts = [entry["text"] for entry in data_group["dialogue"]]
        embeddings = model.encode(texts, batch_size=batch_size, convert_to_tensor=True)
        dialogue_embeddings = [
            {"role": entry["role"], "text": entry["text"], "embedding": emb.tolist()}
            for entry, emb in zip(data_group["dialogue"], embeddings)
        ]
        all_dialogue_embeddings.append({"privacy": data_group["privacy"], "dialogue": dialogue_embeddings})
    with open(path_out, "w") as f:
        json.dump(all_dialogue_embeddings, f, indent=2)
    print("saved", path_out)


In [ ]:
for name in DATASETS:
    path_in = f"{DATA_DIR}/{name}_single_text.json"
    convert(victim_model, path_in, f"{DATA_DIR}/{name}_embed.json")       # victim (e5-base-v2)
    convert(local_model,  path_in, f"{DATA_DIR}/{name}_localembed.json")  # local (all-mpnet-base-v2)
